# Health Risk Prediction Strategy – Modeling Plan

## Objective
We aim to build **three separate predictive models** using our cleaned and transformed dataset. Each model targets a major health condition and is designed to follow a logical, clinical progression. The goal is to avoid data leakage, maximize interpretability, and enable step-by-step risk inference for users.

---

## Multi-Stage Modeling Pipeline

### Model 1: Predict **“Has a high blood pressure”**
- **Target Variable**: `Has a high blood pressure`
- **Excluded Features**:
  - `Has diabetes` → a downstream condition, could induce leakage
  - `Cardiovascular condition (Heart disease or stroke)` → more severe/linked condition
  - `High blood pressure - took medication - 1 month` → treatment indicator, not known beforehand
- **Why**:
  - These are potential outcomes or effects of blood pressure.
  - We want to predict BP status using only upstream lifestyle and health indicators.

---

### Model 2: Predict **“Has diabetes”**
- **Target Variable**: `Has diabetes`
- **Excluded Features**:
  - `Cardiovascular condition (Heart disease or stroke)` → downstream outcome, may cause leakage
- **Included Features**:
  - Keep `Has a high blood pressure` and its medication info, as they can precede or co-occur with diabetes.
- **Why**:
  - High BP and related features can be strong predictors for diabetes.
  - But heart disease is likely a consequence, not a cause.

---

### Model 3: Predict **“Cardiovascular condition (Heart disease or stroke)”**
- **Target Variable**: `Cardiovascular condition (Heart disease or stroke)`
- **Excluded Features**: *None*
- **Why**:
  - This is the final/most severe condition in our cascade.
  - It makes sense to use **all available features**, including blood pressure and diabetes.

---

## Design Intuition

- We build a **sequential prediction pipeline**, where:
  1. Model 1 predicts Blood Pressure.
  2. Model 2 predicts Diabetes using BP status.
  3. Model 3 predicts Heart Disease using all health indicators.
- This mimics **real-world diagnostic flow** and makes the system usable even if a user doesn’t know their exact condition.

---

## Bonus (Future Scope)

- Chain model predictions into each other for inference:
  - Model 1's output → used as input in Model 2
  - Model 2’s output → used in Model 3

This becomes a **progressive health risk assessment tool**, ideal for real-world screening or early intervention systems.

---

## Summary

| Model | Target | Excluded Features |
|-------|--------|-------------------|
| 1 | `Has a high blood pressure` | `Has diabetes`, `Cardiovascular condition`, `High BP - took medication` |
| 2 | `Has diabetes` | `Cardiovascular condition` |
| 3 | `Cardiovascular condition (Heart disease or stroke)` | *None* |

This approach prevents leakage, respects medical logic, and provides robust modeling paths.



In [1]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)


In [2]:
import pandas as pd

# Load the transformed full dataset
df = pd.read_csv(os.path.join(STATS_DIR, "final_cleaned_data.csv"))

# === Model 1: Predicting High Blood Pressure ===
exclude_bp = [
    "Has diabetes",
    "Cardiovascular condition (Heart disease or stroke)",
    "High blood pressure - took medication - 1 month"
]
model1_df = df.drop(columns=exclude_bp)

# === Model 2: Predicting Diabetes ===
exclude_diabetes = [
    "Cardiovascular condition (Heart disease or stroke)"
]
model2_df = df.drop(columns=exclude_diabetes)

# === Model 3: Predicting Cardiovascular Condition ===
# No exclusions for this one
model3_df = df.copy()

# Save datasets for each model
model1_df.to_csv(os.path.join(STATS_DIR, "model1_high_bp.csv"), index=False)
model2_df.to_csv(os.path.join(STATS_DIR, "model2_diabetes.csv"), index=False)
model3_df.to_csv(os.path.join(STATS_DIR, "model3_cardio.csv"), index=False)

print("All 3 model datasets created and saved:")
print("- model1_high_bp.csv")
print("- model2_diabetes.csv")
print("- model3_cardio.csv")


All 3 model datasets created and saved:
- model1_high_bp.csv
- model2_diabetes.csv
- model3_cardio.csv
